# Clase 18: Feature engineering — crear mejores variables

Los modelos de machine learning aprenden mejor cuando las variables son claras,
relevantes y bien preparadas.

**Dataset principal:** `ventas_tienda.csv`

**Pregunta guía:** ¿Cómo transformar los datos crudos de ventas en variables
que un modelo de machine learning pueda usar para predecir si una venta será alta o baja?

In [ ]:
# Qué hace: importar librerías y cargar el dataset principal
# Por qué sirve: sklearn.preprocessing contiene las herramientas de escalado que usaremos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sns.set_theme(style="whitegrid", palette="muted")

ventas = pd.read_csv("../../datasets/ventas_tienda.csv")
estudiantes = pd.read_csv("../../datasets/estudiantes.csv")

print("Shape original:", ventas.shape)
print(ventas.dtypes)
print()
print(ventas.head())

In [ ]:
# Qué hace: exploración inicial para entender qué transformaciones son necesarias
# Por qué sirve: no se puede hacer feature engineering sin entender primero los datos

print("=== VALORES NULOS ===")
print(ventas.isnull().sum())
print()

print("=== ESTADÍSTICAS BÁSICAS ===")
print(ventas.describe().round(2))
print()

print("=== VALORES ÚNICOS EN COLUMNAS CATEGÓRICAS ===")
categoricas = ventas.select_dtypes(include="object").columns
for col in categoricas:
    print(f"{col}: {ventas[col].unique()[:10]}")

In [ ]:
# Qué hace: crear variables numéricas derivadas combinando columnas existentes
# Por qué sirve: captura relaciones que el modelo no vería fácilmente por separado

# Copia del dataframe para no modificar el original
df = ventas.copy()

# Ingreso bruto: cuánto se vendió antes de descuentos
df["ingreso_bruto"] = df["precio_unitario"] * df["unidades"]

# Ingreso neto: cuánto se cobró realmente después del descuento
df["ingreso_neto"] = df["ingreso_bruto"] * (1 - df["descuento"] / 100)

# Ganancia por unidad: precio de venta menos costo
df["ganancia_por_unidad"] = df["precio_unitario"] - df["costo_unitario"]

# Margen porcentual: qué porcentaje del precio es ganancia
# Evitamos dividir entre cero con un pequeño truco
df["margen_pct"] = np.where(
    df["precio_unitario"] > 0,
    df["ganancia_por_unidad"] / df["precio_unitario"] * 100,
    0
)

print(df[["precio_unitario", "unidades", "descuento", "ingreso_bruto", "ingreso_neto", "margen_pct"]].head(8).round(2))

In [ ]:
# Qué hace: extraer información temporal desde la columna de fecha
# Por qué sirve: las fechas crudas no son útiles para modelos — sus componentes sí lo son

# Convertir a tipo datetime
df["fecha"] = pd.to_datetime(df["fecha"])

# Extraer componentes temporales
df["mes"]           = df["fecha"].dt.month          # 1 a 12
df["dia_semana"]    = df["fecha"].dt.dayofweek       # 0=Lunes, 6=Domingo
df["dia_mes"]       = df["fecha"].dt.day             # 1 a 31
df["trimestre"]     = df["fecha"].dt.quarter         # 1 a 4
df["es_fin_semana"] = (df["dia_semana"] >= 5).astype(int)  # 1 si sábado o domingo

print("Distribución de ventas por mes:")
print(df["mes"].value_counts().sort_index())
print()
print(f"Ventas en fin de semana: {df['es_fin_semana'].sum()} ({df['es_fin_semana'].mean()*100:.1f}%)")
print(f"Ventas en días de semana: {(1-df['es_fin_semana']).sum()}")

In [ ]:
# Qué hace: codificar variables categóricas con one-hot encoding
# Por qué sirve: los modelos de ML necesitan números, no texto

print("Categorías en 'categoria':", df["categoria"].unique())
print()

# One-hot encoding: crea una columna binaria por cada categoría
# drop_first=True elimina una columna para evitar la trampa de multicolinealidad perfecta
dummies_cat = pd.get_dummies(df["categoria"], prefix="cat", drop_first=True)
df = pd.concat([df, dummies_cat], axis=1)

print("Columnas creadas por one-hot:")
print(dummies_cat.columns.tolist())
print()
print(dummies_cat.head(5))

In [ ]:
# Qué hace: codificar variables ordinales con map y crear binning
# Por qué sirve: algunas categorías tienen orden natural; otras se benefician de agruparse en rangos

# Codificación ordinal: supone que hay una columna 'canal' con orden implícito
if "canal" in df.columns:
    mapa_canal = {"online": 0, "tienda": 1, "mayorista": 2}
    df["canal_cod"] = df["canal"].map(mapa_canal)
    print("Canal codificado:", df["canal_cod"].value_counts())

# Binning con pd.cut: rangos definidos manualmente por conocimiento del negocio
df["rango_ingreso"] = pd.cut(
    df["ingreso_neto"],
    bins=[0, 500, 1500, 5000, float("inf")],
    labels=["bajo", "medio", "alto", "premium"]
)

# Binning con pd.qcut: rangos basados en cuartiles (igual cantidad de registros)
df["cuartil_ingreso"] = pd.qcut(df["ingreso_neto"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

print("\nDistribución por rango de ingreso:")
print(df["rango_ingreso"].value_counts())

print("\nDistribución por cuartil:")
print(df["cuartil_ingreso"].value_counts().sort_index())

In [ ]:
# Qué hace: escalar variables numéricas con StandardScaler y MinMaxScaler
# Por qué sirve: evita que variables con magnitudes grandes dominen sobre las pequeñas

# Variables numéricas para escalar
cols_escalar = ["precio_unitario", "unidades", "ingreso_neto", "descuento", "margen_pct"]
df_num = df[cols_escalar].fillna(0)  # por si hay nulos

# StandardScaler: media=0, desviación estándar=1
# Úsalo cuando los datos siguen una distribución aproximadamente normal
scaler_std = StandardScaler()
datos_std = scaler_std.fit_transform(df_num)
df_std = pd.DataFrame(datos_std, columns=[c + "_std" for c in cols_escalar])

# MinMaxScaler: rango [0, 1]
# Úsalo cuando necesitas una cota absoluta o hay outliers fuertes que quieres conservar
scaler_mm = MinMaxScaler()
datos_mm = scaler_mm.fit_transform(df_num)
df_mm = pd.DataFrame(datos_mm, columns=[c + "_mm" for c in cols_escalar])

print("=== ESTADÍSTICAS DESPUÉS DE STANDARD SCALER ===")
print(df_std.describe().round(3))
print()
print("=== ESTADÍSTICAS DESPUÉS DE MINMAX SCALER ===")
print(df_mm.describe().round(3))

In [ ]:
# Qué hace: crear variables de interacción que capturan efectos conjuntos
# Por qué sirve: algunas relaciones solo existen cuando dos variables interactúan

# Interacción precio × descuento: el impacto real del descuento depende del precio
df["precio_x_descuento"] = df["precio_unitario"] * df["descuento"]

# Interacción unidades × margen: total de ganancia generado
df["ganancia_total"] = df["unidades"] * df["ganancia_por_unidad"]

# Variable objetivo: ¿fue una venta alta? (sobre la mediana)
umbral = df["ingreso_neto"].median()
df["venta_alta"] = (df["ingreso_neto"] > umbral).astype(int)

print(f"Umbral para 'venta alta': {umbral:.2f}")
print(f"Ventas altas: {df['venta_alta'].sum()} ({df['venta_alta'].mean()*100:.1f}%)")
print(f"Ventas bajas: {(df['venta_alta']==0).sum()} ({(1-df['venta_alta'].mean())*100:.1f}%)")

In [ ]:
# Qué hace: calcular correlaciones de todas las variables numéricas con el target
# Por qué sirve: selecciona las variables más informativas y descarta el ruido

# Solo columnas numéricas
df_num_completo = df.select_dtypes(include="number")

# Correlación con la variable objetivo
correlaciones_target = df_num_completo.corr()["venta_alta"].drop("venta_alta")
correlaciones_target = correlaciones_target.sort_values(key=abs, ascending=False)

print("=== CORRELACIÓN CON 'venta_alta' (ordenada por importancia) ===")
print(correlaciones_target.round(3))

# Variables con correlación absoluta mayor a 0.1
vars_seleccionadas = correlaciones_target[correlaciones_target.abs() > 0.1].index.tolist()
print(f"\nVariables seleccionadas ({len(vars_seleccionadas)}): {vars_seleccionadas}")

In [ ]:
# Qué hace: visualizar las correlaciones con un barplot horizontal
# Por qué sirve: facilita identificar visualmente las variables más y menos relevantes

# Top 15 variables por correlación absoluta con el target
top_vars = correlaciones_target.abs().sort_values(ascending=True).tail(15)

colores = ["steelblue" if v > 0 else "coral" for v in correlaciones_target[top_vars.index]]

plt.figure(figsize=(9, 6))
bars = plt.barh(top_vars.index, top_vars.values, color=colores)
plt.axvline(x=0.1, color="red", linestyle="--", linewidth=1, label="Umbral 0.1")
plt.title("Correlación (absoluta) de variables con 'venta_alta'")
plt.xlabel("Correlación absoluta")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Qué hace: construir el dataset final listo para modelos de machine learning
# Por qué sirve: consolida todo el trabajo de feature engineering en un DataFrame limpio

# Seleccionar las variables numéricas finales + target
features_finales = [v for v in vars_seleccionadas if v in df.columns][:12]  # top 12
features_finales.append("venta_alta")  # agregar el target

df_final = df[features_finales].dropna()

print("=== DATASET FINAL PARA ML ===")
print(f"Shape: {df_final.shape}")
print(f"Columnas: {df_final.columns.tolist()}")
print()
print(df_final.head(5).round(2))
print()
print("Distribución del target:")
print(df_final["venta_alta"].value_counts())
print()
print("El dataset está listo para ser dividido en train/test y entregado a un modelo.")

## Resumen de técnicas de feature engineering

| Técnica | Herramienta | Cuándo usarla |
|---|---|---|
| Variables derivadas | Operaciones aritméticas | Cuando dos columnas combinadas capturan mejor una relación |
| Extracción de fechas | `dt.month`, `dt.dayofweek` | Cuando la temporalidad puede predecir el target |
| One-hot encoding | `pd.get_dummies` | Variables categóricas sin orden natural |
| Codificación ordinal | `.map()` | Variables categóricas con orden (bajo/medio/alto) |
| Binning | `pd.cut` / `pd.qcut` | Convertir continuas en grupos cuando la magnitud exacta no importa |
| StandardScaler | `StandardScaler` | Cuando los datos son aproximadamente normales |
| MinMaxScaler | `MinMaxScaler` | Cuando se necesita un rango [0,1] fijo |
| Interacciones | Multiplicación de columnas | Cuando el efecto de una variable depende de otra |
| Selección | Correlación con target | Para reducir ruido y redundancia antes del modelo |

**Regla de oro:** Un modelo sencillo con buenas variables supera a un modelo complejo con variables malas.

**Próximas clases:** Con este dataset preparado, estarás listo para introducir los primeros modelos de machine learning supervisado: regresión logística, árboles de decisión y más.